# LangChain Run Collector Tracer Reference

Developer-facing statements defined in `langchain_core.tracers.run_collector`.

# `RunCollectorCallbackHandler: BaseTracer`

Concrete synchronous tracer that collects completed root traces for inspection and evaluation.

Each collected root trace retains its nested child-run tree.

## Fields

```python
name: str = "run-collector_callback_handler" # Callback-handler name
example_id: UUID | None # Example identifier assigned to collected root runs
traced_runs: list[Run] # Copies of completed root runs collected by the handler
```

## Constructor

```python
RunCollectorCallbackHandler(
    example_id: UUID | str | None = None, # Example identifier assigned to collected runs
    **kwargs: Any, # Arguments forwarded to BaseTracer
) -> None
```

A string `example_id` is converted to `UUID`. The `traced_runs` list is initialized empty.

## Behaviour

The handler uses the lifecycle callbacks inherited from `BaseTracer`.

When a root run finishes, the handler copies the completed `Run`, sets the copy's `reference_example_id` to `example_id`, and appends the copy to `traced_runs`.

Child runs are not appended as separate top-level entries. They remain available through the collected root run's nested child-run structure.

In [ ]:
from langchain_core.runnables import RunnableLambda # Import the real LangChain runnable
from langchain_core.tracers.run_collector import RunCollectorCallbackHandler # Import the run collector


def add_two(number: int) -> int: # Define the first operation
    return number + 2 # Add two to the input


def multiply_by_three(number: int) -> int: # Define the second operation
    return number * 3 # Multiply the input by three


add_step = RunnableLambda(add_two).with_config( # Create the first child runnable
    run_name="add_two"
)

multiply_step = RunnableLambda(multiply_by_three).with_config( # Create the second child runnable
    run_name="multiply_by_three"
)

pipeline = add_step | multiply_step # Combine both operations into one pipeline

collector = RunCollectorCallbackHandler() # Create the real run collector

result = pipeline.invoke( # Run the pipeline
    5, # Provide the input
    config={ # Configure tracing
        "callbacks": [collector], # Attach the run collector
        "run_name": "math_pipeline", # Name the root run
        "tags": ["math", "demo"], # Add tags
        "metadata": {"source": "jupyter"}, # Add metadata
    },
) # Finish invoking the pipeline

print("Pipeline result:", result) # Display (5 + 2) * 3
print("Collected root runs:", len(collector.traced_runs)) # Display the root-run count

root_run = collector.traced_runs[0] # Access the collected root run

print("\nRoot run name:", root_run.name) # Display the root run name
print("Root run type:", root_run.run_type) # Display the root run type
print("Root inputs:", root_run.inputs) # Display the recorded input
print("Root outputs:", root_run.outputs) # Display the recorded output
print("Root tags:", root_run.tags) # Display the run tags
print("Child runs:", len(root_run.child_runs)) # Display the number of child runs

print("\nChild-run details:") # Display a heading

for child_run in root_run.child_runs: # Visit every nested child run
    print("Name:", child_run.name) # Display the child run name
    print("Type:", child_run.run_type) # Display the child run type
    print("Inputs:", child_run.inputs) # Display the child input
    print("Outputs:", child_run.outputs) # Display the child output
    print() # Print a blank line